# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice: Line chart.** A line chart is preferred as the question is about the change of each continent through a few years. The connection of the points makes it easy to compare the direction and rate of growth. I index all continents to 100 in 1970, so the lines represent relative growth rather than the very different population sizes. The title highlights the main finding: Africa saw the fastest growth, reaching about 390 on the index (about 3.9 times its 1970 population) by 2022.

In [9]:
# Q1 — Relative population growth (1970 = 100)
year_cols = [f'{y} Population' for y in [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]]
continent_year = df.groupby('Continent')[year_cols].sum()
growth_index = continent_year.div(continent_year['1970 Population'], axis=0) * 100
growth_long = (growth_index.rename(columns=lambda c: int(c[:4]))
               .reset_index().melt(id_vars='Continent', var_name='Year', value_name='Index'))

fig = px.line(
    growth_long, x='Year', y='Index', color='Continent',
    markers=True,
    title='Africa’s population grew fastest — to 3.9× its 1970 size by 2022',
    labels={'Index': 'Population index (1970 = 100)', 'Year': 'Year', 'Continent': 'Continent'},
    color_discrete_map={'Africa': '#8B0000', 'Asia': '#777777', 'Europe': '#A6A6A6',
                        'North America': '#BDBDBD', 'South America': '#D0D0D0', 'Oceania': '#929292'}
)
fig.update_layout(template='plotly_white', legend_title_text='', yaxis=dict(showgrid=False),
                  xaxis=dict(showgrid=False), hovermode='x unified')
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice: Sorted horizontal bar chart.**A sorted bar chart makes it simple to rank and compare the 15 largest absolute population increases. Country names are displayed in horizontal bars, color highlights African countries, the other continents are gray. Four of the 15 countries are from Africa.

In [10]:
# Q2 — Top 15 countries by absolute population added, 1970–2022
q2 = df.copy()
q2['People added'] = q2['2022 Population'] - q2['1970 Population']
top15 = q2.nlargest(15, 'People added').sort_values('People added', ascending=True)
top15['Highlight'] = np.where(top15['Continent'].eq('Africa'), 'Africa', 'Other continents')
africa_count = int(top15['Continent'].eq('Africa').sum())

fig = px.bar(
    top15, x='People added', y='Country', orientation='h', color='Highlight',
    color_discrete_map={'Africa': '#8B0000', 'Other continents': '#B8B8B8'},
    text='People added',
    title=f'The 15 largest population gains include {africa_count} African countries',
    labels={'People added': 'Population added (people)', 'Country': '', 'Highlight': ''}
)
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside', cliponaxis=False)
fig.update_layout(template='plotly_white', showlegend=True, legend_title_text='',
                  xaxis=dict(showgrid=False, tickformat=','),
                  yaxis=dict(showgrid=False), margin=dict(r=110))
fig.show()
print(f'African countries among the top 15: {africa_count}')

African countries among the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice: Scatter plot.** Land area and population are both numerical data, so a scatter plot would be a good way to show the relationship and to see which countries do not fit with the others. Logarithmic axes allow reading countries of very different sizes Continent color helps to compare regional patterns In this data area population and area are only moderately associated. Large area does not mean large population automatically.

In [4]:
# Q3 — Land area versus 2022 population
# Label the most populous countries to make the main outliers easy to identify.
label_countries = ['China', 'India', 'United States', 'Indonesia', 'Pakistan',
                   'Nigeria', 'Brazil', 'Bangladesh']
q3 = df.copy()
q3['Label'] = np.where(q3['Country'].isin(label_countries), q3['Country'], '')
fig = px.scatter(
    q3, x='Area', y='2022 Population', color='Continent', size='World share (%)',
    size_max=28, hover_name='Country', text='Label',
    log_x=True, log_y=True,
    title='Land area and population are only moderately related',
    labels={'Area': 'Land area (km², log scale)',
            '2022 Population': '2022 population (log scale)', 'Continent': 'Continent',
            'World share (%)': 'World share (%)'}
)
fig.update_traces(textposition='top center', marker=dict(line=dict(width=0.5, color='white')))
fig.update_layout(template='plotly_white', legend_title_text='',
                  xaxis=dict(showgrid=False), yaxis=dict(showgrid=False))
fig.show()
print(f"Pearson correlation (raw values): {df[['Area', '2022 Population']].corr().iloc[0, 1]:.2f}")

Pearson correlation (raw values): 0.45


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice: Violin plot.** A violin plot shows the full distribution of country-level population density within each continent, rather than only a mean or median. The logarithmic vertical axis makes the highly skewed density values easier to compare; the embedded box plot gives a compact view of the median and interquartile range.

In [5]:
# Q4 — Distribution of population density by continent
fig = px.violin(
    df, x='Continent', y='Density', color='Continent',
    box=True, points='outliers', log_y=True,
    title='Country-level population density varies widely within each continent',
    labels={'Density': 'People per km² (log scale)', 'Continent': ''}
)
fig.update_layout(template='plotly_white', showlegend=False,
                  xaxis=dict(showgrid=False), yaxis=dict(showgrid=False))
fig.update_traces(meanline_visible=True, line_color='#555555')
fig.show()
print('Median density by continent (people/km²):')
print(df.groupby('Continent')['Density'].median().sort_values(ascending=False).round(1))

Median density by continent (people/km²):
Continent
North America    221.4
Asia             123.0
Europe           106.9
Oceania           78.2
Africa            59.7
South America     17.8
Name: Density, dtype: float64


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice: Choropleth map.** The choropleth is the best fit because the question is asking about the geography of population growth and decline. Countries are colored by percentage change from 2000 to 2022, using a diverging color scale centered on zero so that growth and shrinkage are immediately apparent. Gray: No geographic matching or geographic matching not available.

In [6]:
# Q5 — Percentage population change, 2000–2022
q5 = df.copy()
q5['Change (%)'] = (q5['2022 Population'] - q5['2000 Population']) / q5['2000 Population'] * 100
fig = px.choropleth(
    q5, locations='CCA3', locationmode='ISO-3', color='Change (%)',
    hover_name='Country', hover_data={'Change (%)': ':.1f', 'CCA3': False},
    color_continuous_scale='RdBu', color_continuous_midpoint=0,
    title='Population growth dominates globally, while several countries have shrunk',
    labels={'Change (%)': 'Population change (%)'}
)
fig.update_layout(template='plotly_white', geo=dict(showframe=False, showcoastlines=False,
                                                   showcountries=True, showland=True,
                                                   landcolor='#EEEEEE'),
                  coloraxis_colorbar_title='Change (%)')
fig.show()
print(f"Countries that grew: {(q5['Change (%)'] > 0).sum()}")
print(f"Countries that shrank: {(q5['Change (%)'] < 0).sum()}")
print('Largest percentage declines:')
print(q5.nsmallest(10, 'Change (%)')[['Country', 'Continent', 'Change (%)']].to_string(index=False))

Countries that grew: 200
Countries that shrank: 34
Largest percentage declines:
                 Country Continent  Change (%)
Northern Mariana Islands   Oceania  -38.321840
          American Samoa   Oceania  -23.968745
               Lithuania    Europe  -23.601880
        Marshall Islands   Oceania  -23.338374
                 Moldova    Europe  -23.016822
                  Latvia    Europe  -22.648786
  Bosnia and Herzegovina    Europe  -22.630888
            Vatican City    Europe  -21.658986
       Wallis and Futuna   Oceania  -21.401888
                 Ukraine    Europe  -18.776723


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice: Treemap.** A treemap shows two levels of nesting in one figure: first the continent, then each country's contribution within that continent. The area of ​​the rectangle encodes 2022 population so that the continental totals andthe largest countries are both visible at once. Hover labels provide exact values ​​without cluttering the figure.

In [7]:
# Q6 — 2022 population by continent and country
fig = px.treemap(
    df, path=[px.Constant('World'), 'Continent', 'Country'],
    values='2022 Population', color='Continent',
    color_discrete_map={'Africa': '#2878B5', 'Asia': '#777777', 'Europe': '#A6A6A6',
                        'North America': '#BDBDBD', 'South America': '#D0D0D0', 'Oceania': '#929292'},
    title='Asia contains the largest share of the world’s population',
    hover_data={'2022 Population': ':,'}
)
fig.update_traces(textinfo='label+percent parent', root_color='white')
fig.update_layout(template='plotly_white', margin=dict(t=65, l=10, r=10, b=10))
fig.show()
print('2022 population by continent:')
print(df.groupby('Continent')['2022 Population'].sum().sort_values(ascending=False).to_string())

2022 population by continent:
Continent
Asia             4721383274
Africa           1426730932
Europe            743147538
North America     600296136
South America     436816608
Oceania            45038554


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice: 100% stacked bar chart.** Here, a 100% stacked bar chart is appropriate since the question asks us to compare the share of each continent of the world total over time. Each year is 100 %. The world population is increasing but we can see the change in composition . Africa, Asia and Europe are displayed separately, the other three continents are combined as “Rest of the world”.

In [8]:
# Q7 — World population shares over time
years = [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]
year_cols = [f'{y} Population' for y in years]
continent_totals = df.groupby('Continent')[year_cols].sum()
world_totals = continent_totals.sum(axis=0)
share = continent_totals.div(world_totals, axis=1) * 100

focus = share.loc[['Africa', 'Asia', 'Europe']].copy()
focus.loc['Rest of the world'] = share.drop(index=['Africa', 'Asia', 'Europe']).sum(axis=0)
share_long = (focus.rename(columns=lambda c: int(c[:4])).reset_index()
              .melt(id_vars='Continent', var_name='Year', value_name='World share (%)'))

fig = px.bar(
    share_long, x='Year', y='World share (%)', color='Continent',
    barmode='stack',
    category_orders={'Continent': ['Africa', 'Asia', 'Europe', 'Rest of the world']},
    color_discrete_map={'Africa': '#2878B5', 'Asia': '#777777',
                        'Europe': '#A6A6A6', 'Rest of the world': '#D0D0D0'},
    title='Africa’s share is rising while Europe’s share is falling',
    labels={'Year': 'Year', 'World share (%)': 'Share of world population (%)',
            'Continent': ''}
)
fig.update_layout(template='plotly_white', legend_title_text='',
                  yaxis=dict(range=[0, 100], ticksuffix='%', showgrid=False),
                  xaxis=dict(showgrid=False), bargap=0.18)
fig.show()
print('Share change, 1970 to 2022 (percentage points):')
for continent in ['Africa', 'Asia', 'Europe']:
    print(f"{continent}: {share.loc[continent, '2022 Population'] - share.loc[continent, '1970 Population']:+.1f} pp")

Share change, 1970 to 2022 (percentage points):
Africa: +8.0 pp
Asia: +1.2 pp
Europe: -8.4 pp
